<a href="https://colab.research.google.com/github/gonmpa/Notebook_Jupyter_Mineria_Datos_Supervisado_Clasificacion/blob/main/Evaluacion_Final_MD_Pino_Gonzalo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Portada: Predicción de Tramos de Carrera Docente (Evaluación Docente 2024)

| Campo | Detalle |
| :--- | :--- |
| **Asignatura** | Minería de Datos (`IEI097`) |
| **Proyecto** | Implementación CRISP-DM: Clasificación de Tramos de Carrera Docente (`PF_CAT_CARRERA`) |
| **Integrantes** | Gonzalo Matías Pino Araya  |
| **Grupo 00 / Experimento MLflow** | `IEI097-G00`  |
| **Repositorio GitHub** | [https://github.com/gonmpa/Notebook_Jupyter_Mineria_Datos_Supervisado_Clasificacion](https://github.com/gonmpa/Notebook_Jupyter_Mineria_Datos_Supervisado_Clasificacion/tree/main) |
| **Informe CRISP-DM (Act. 2.13)** | [https://github.com/gonmpa/Notebook_Jupyter_Mineria_Datos_Supervisado_Clasificacion/blob/main/IEI097_Informe_G00.pdf](https://github.com/gonmpa/Notebook_Jupyter_Mineria_Datos_Supervisado_Clasificacion/blob/main/IEI097_Informe_G00.pdf) |

### Resumen del Problema de Negocio
El Sistema de Desarrollo Profesional Docente en Chile clasifica a los profesores evaluados en cinco tramos (`A`, `B`, `C`, `D`, `E`) según su desempeño. El objetivo de este proyecto es construir y evaluar modelos de aprendizaje supervisado capaces de predecir el tramo asignado (`PF_CAT_CARRERA`) a partir de variables sociodemográficas, laborales y del establecimiento educacional, contrastando los resultados con las hipótesis formuladas en la fase de planificación CRISP-DM.

## 1. Carga de Datos

Para garantizar la reproducibilidad completa del notebook sin intervención manual (evitando el uso de `files.upload()`), el conjunto de datos oficial del Ministerio de Educación (`20251118_EVALUACION_DOCENTE_2024_20251118_PUBL.csv`) se descarga directamente desde su fuente alojada en repositorio público.

### 1.1 Inspección Inicial de Dimensiones y Tipos de Datos

A continuación se verifican las dimensiones del dataset (filas y columnas), los tipos de datos inferidos por `pandas` y una muestra inicial de los registros.

#### Interpretación 1.1 — Estructura Inicial del Dataset
* **Dimensiones:** El conjunto de datos cuenta con `25.197` registros (docentes evaluados en el proceso 2024) y `22` variables.
* **Tipos de datos:** Se identifican variables numéricas enteras (códigos territoriales, bienios de experiencia, fechas en formato `AAAAMM`), variables decimales (puntaje de portafolio) y variables categóricas de texto (tramo de carrera, dependencia, ruralidad).
* **Observación preliminar:** *[Completar tras ejecutar C-1.3 respecto a formatos de fechas o códigos que requieren conversión].*

## 2. Análisis Exploratorio de Datos (EDA)

En esta fase se examina la calidad, distribución y relación entre las variables del dataset con el propósito de fundamentar técnicamente cada decisión de limpieza y transformación en el `Pipeline` de preparación.

### 2.1 Estadísticos Descriptivos
Se calculan las medidas de tendencia central, dispersión y cuartiles para las variables numéricas, así como el conteo de categorías únicas y frecuencias modales para las variables categóricas.

#### Interpretación 2.1 — Estadísticos Descriptivos
* **Variables numéricas:** *[Comentar rangos de bienios de experiencia, edad/año de nacimiento y dispersión observada].*
* **Variables categóricas:** *[Comentar categorías predominantes en dependencia del establecimiento, género y ruralidad].*

### 2.2 Calidad de Datos: Valores Nulos, Duplicados y Centinelas
Se audita la presencia de registros duplicados, el porcentaje de valores faltantes (`NaN`) por columna y la existencia de valores centinela codificados en el diccionario del Mineduc (como `190001` en fecha de nacimiento).

### 2.3 Distribución de la Variable Objetivo (`PF_CAT_CARRERA`)
Al tratarse de un problema de clasificación multiclase, se analiza la frecuencia absoluta y porcentual de cada tramo de carrera docente (`A`, `B`, `C`, `D`, `E`) para evaluar el grado de desbalanceo de clases.

#### Interpretación 2.3 — Distribución de la Variable Objetivo
* **Desbalanceo de clases:** Se observa una distribución fuertemente asimétrica: los tramos intermedios (*[indicar cuáles tras graficar, ej. Temprano/Avanzado]*) concentran la mayor proporción de docentes, mientras que los tramos extremos (*[ej. Experto II]*) representan una fracción minoritaria.
* **Impacto en el modelado:** Este desbalanceo exige utilizar **partición estratificada** (`stratify=y`), activar pesos balanceados (`class_weight='balanced'`) en los algoritmos y priorizar métricas como **F1-Score Macro** por sobre el *Accuracy* global.

### 2.4 Distribuciones de Variables Predictoras (Numéricas y Categóricas)
Se visualizan histogramas para las variables numéricas clave y gráficos de frecuencia para las variables categóricas relevantes del contexto laboral docente.

#### Interpretación 2.4 — Distribuciones de Predictoras
* **Asimetrías y sesgos:** *[Describir si variables como bienios o edad presentan sesgo positivo/negativo y si requieren estandarización].*
* **Concentración categórica:** *[Describir la distribución por dependencia administrativa, área geográfica urbana/rural y género].*

### 2.5 Detección de Valores Atípicos (*Outliers*) mediante Box-plots
Se construyen diagramas de caja (*box-plots*) sobre las variables numéricas continuas y discretas para identificar valores extremos y evaluar su coherencia con la realidad del sistema escolar chileno.

#### Interpretación 2.5 — Análisis de Outliers
* **Hallazgos en box-plots:** *[Indicar qué variables muestran puntos fuera de los bigotes intercuartílicos, ej. docentes con alta cantidad de bienios o edades avanzadas en ejercicio].*
* **Decisión técnica:** Dado que estos valores extremos representan trayectorias docentes reales y no errores de digitación (una vez limpiado el centinela `190001`), se conservarán aplicando `StandardScaler` para estabilizar su escala.

### 2.6 Matriz de Correlaciones y Detección de Fuga de Datos (*Data Leakage*)
Se analiza la matriz de correlaciones entre variables numéricas y se examina la relación entre el puntaje de portafolio (`PF_PJE`) y la categoría asignada (`PF_CAT_CARRERA`).

#### Interpretación 2.6 — Correlaciones y Fuga de Datos (`PF_PJE`)
* **Colinealidad y fuga de datos:** Se comprueba visual y numéricamente que `PF_CAT_CARRERA` es la discretización directa por umbrales fijos de `PF_PJE`. Incluir `PF_PJE` como predictor provocaría **fuga de datos (*data leakage*)**, generando un rendimiento artificial cercano al 100% e impidiendo evaluar el peso real de las variables contextuales y laborales. Por ello, `PF_PJE` se excluye de la matriz $X$.
* **Correlaciones entre predictoras:** *[Comentar correlación esperada entre edad y bienios de experiencia, y redundancia entre códigos y nombres de regiones/dependencias].*

### 2.7 Síntesis del EDA y Decisiones de Preparación

| Hallazgo en el EDA | Variable(s) Afectada(s) | Acción en Fase 3 (Preparación / Pipeline) |
| :--- | :--- | :--- |
| Fuga de datos determinística | `PF_PJE` | Exclusión de la matriz de características $X$. |
| 100% de valores nulos | `PF_ESC` | Eliminación de la columna. |
| Identificador único sin valor predictivo | `MRUN` | Eliminación para evitar memorización (*overfitting*). |
| Redundancia texto-código | `NOM_REG_RBD`, `NOM_COM_RBD`, etc. | Conservar solo códigos numéricos (`COD_*`) y eliminar etiquetas duplicadas. |
| Valor centinela `190001` | `FEC_NAC_DOC` | Cálculo de edad a 2024, reemplazo de `190001` por `NaN` e imputación por mediana en `Pipeline`. |
| Desbalanceo de clases (A–E) | `PF_CAT_CARRERA` | Filtrado de nulos, `stratify=y` en `train_test_split` y `class_weight='balanced'`. |

## 3. Preparación de los Datos

Todas las transformaciones estadísticas (imputación, escalamiento y codificación *One-Hot*) se encapsulan dentro de un `Pipeline` de `scikit-learn` ajustado exclusivamente con el conjunto de entrenamiento para evitar cualquier fuga de datos hacia el conjunto de prueba.

### 3.1 Limpieza Estructural y Selección de Variables
Se aplican las reglas determinísticas previas a la partición: filtrado de registros sin categoría válida en `PF_CAT_CARRERA`, derivación de la variable `EDAD` y eliminación de columnas descartadas (`PF_PJE`, `PF_ESC`, `MRUN` y descripciones redundantes).

### 3.2 Partición Estratificada (`Train` / `Test`)
Se dividen los datos en **80% entrenamiento** y **20% prueba** fijando la semilla `random_state=42` y manteniendo la proporción exacta de los tramos A–E mediante `stratify=y`. Esta misma partición se utilizará en los tres modelos.

### 3.3 Construcción del `Pipeline` de Preprocesamiento (`ColumnTransformer`)
Se define un preprocesador unificado que aplica:
* **Variables numéricas:** `SimpleImputer(strategy="median")` seguido de `StandardScaler()`.
* **Variables categóricas:** `SimpleImputer(strategy="most_frequent")` seguido de `OneHotEncoder(handle_unknown="ignore")`.

### 3.4 Validación Matemática sobre Muestra Controlada (Cálculo Manual)
*(Opcional de profundización metodológica)*: Antes de entrenar sobre el total de registros, se extrae una submuestra controlada de docentes para verificar manualmente cómo operan las ecuaciones internas de decisión (función **Softmax** en Regresión Logística e **Impureza de Gini** en árboles de decisión).

## 4. Modelado

Se entrenan tres modelos bajo exactamente el mismo protocolo experimental (misma partición `80/20`, semilla `42` y esquema de validación cruzada estratificada de 5 pliegues `StratifiedKFold(n_splits=5)`):
1. **Modelo Base:** `DummyClassifier(strategy="most_frequent")` como referencia mínima (*benchmark*).
2. **Modelo 1 (Familia Lineal):** `LogisticRegression` (Regresión Logística Multinomial con `class_weight="balanced"`).
3. **Modelo 2 (Familia Ensamble de Árboles):** `RandomForestClassifier` (con `class_weight="balanced"`).

### 4.1 Configuración del Experimento en MLflow
Se inicializa el servidor de seguimiento local y se configura el experimento oficial del grupo (`IEI097-GXX`) para registrar parámetros, métricas, etiquetas (`grupo` y `tipo_modelo`) y artefactos gráficos de cada ejecución.

### 4.2 Modelo Base: `DummyClassifier (most_frequent)`
Este modelo predice siempre la clase mayoritaria del conjunto de entrenamiento. Su función es establecer el piso mínimo de rendimiento que cualquier modelo inteligente debe superar.

### 4.3 Modelo 1 (Familia Lineal): Regresión Logística Multinomial
Se entrena un modelo lineal generalizado multiclase con regularización L2, `max_iter=1000` y ponderación balanceada de clases para evaluar relaciones lineales entre las características docentes y el tramo asignado.

### 4.4 Modelo 2 (Familia Ensamble): Random Forest Classifier
Se entrena un ensamble de árboles de decisión independientes para capturar interacciones no lineales complejas entre experiencia, dependencia y contexto del establecimiento, controlando el sobreajuste mediante `max_depth` y `min_samples_leaf`.

## 5. Evaluación y Comparación de Modelos

En esta sección se consolidan los resultados obtenidos tanto en **validación cruzada (entrenamiento)** como en el **conjunto de prueba (evaluación final)**, analizando el comportamiento por clase y el impacto de los errores en el contexto educativo.

### 5.1 Tabla Comparativa de Métricas
Se comparan `Accuracy`, `Precision Macro`, `Recall Macro` y `F1-Score Macro` (métrica principal dada la asimetría entre tramos).

| Modelo | Familia | F1-Macro (CV Train) | Accuracy (Test) | Precision Macro (Test) | Recall Macro (Test) | F1-Macro (Test) |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **DummyClassifier (Base)** | Referencia | *[0.XX]* | *[0.XX]* | *[0.XX]* | *[0.XX]* | *[0.XX]* |
| **Regresión Logística** | Lineal | *[0.XX]* | *[0.XX]* | *[0.XX]* | *[0.XX]* | *[0.XX]* |
| **Random Forest** | Ensamble | *[0.XX]* | *[0.XX]* | *[0.XX]* | *[0.XX]* | *[0.XX]* |

#### Interpretación 5.1 — Comparación de Rendimiento
* **Superación del modelo base:** *[Analizar cuánto mejoran la Regresión Logística y Random Forest respecto al `DummyClassifier`, especialmente en `F1-Macro`, donde el modelo base colapsa al predecir una sola clase].*
* **Estabilidad (CV vs. Test):** *[Verificar si existe brecha entre el puntaje de validación cruzada y el de prueba para descartar sobreajuste].*

### 5.2 Matrices de Confusión y Gráficos de Apoyo por Modelo
Se visualizan las matrices de confusión normalizadas de los tres modelos sobre el conjunto de prueba, junto con los reportes de clasificación desglosados por tramo (`A`, `B`, `C`, `D`, `E`).

#### Interpretación 5.2 — Análisis de Errores y Costo en el Problema de Negocio
* **Patrón de confusión:** *[Analizar si los errores ocurren entre tramos adyacentes —por ejemplo, confundir tramo C con B o D— o entre tramos extremos].*
* **Costo del error en el negocio:** En el sistema de carrera docente, subestimar el tramo de un profesor (falso negativo en tramos altos como *Experto I/II*) implica invisibilizar factores asociados a la excelencia pedagógica y asignaciones salariales; por otro lado, sobreestimar el tramo puede generar expectativas erróneas en políticas de acompañamiento docente.

### 5.3 Elección Justificada del Mejor Modelo
* **Modelo seleccionado:** **[Indicar modelo ganador, ej. Random Forest Classifier]**.
* **Justificación técnica:** Se selecciona este modelo porque obtiene el mayor equilibrio en **F1-Score Macro** y **Recall Macro** sobre datos no vistos, demuestra capacidad para modelar relaciones no lineales entre variables laborales y territoriales sin presentar sobreajuste severo respecto a la validación cruzada, y logra identificar tramos minoritarios que el modelo base ignora por completo.

### 5.4 Evidencias de Observabilidad en MLflow
A continuación se adjuntan las capturas de la interfaz de MLflow almacenadas en la carpeta `evidencias_mlflow/` del repositorio, acreditando el registro de los runs, parámetros, métricas y artefactos gráficos.

**1. Lista de ejecuciones (`Runs`) del experimento `IEI097-GXX`:**
![Lista de Runs MLflow](evidencias_mlflow/01_lista_runs.png)

**2. Comparación de métricas entre modelos (`Compare`):**
![Comparación de Runs MLflow](evidencias_mlflow/02_comparacion_runs.png)

**3. Detalle del mejor `Run` con parámetros, métricas y artefactos registrados:**
![Detalle de Run MLflow](evidencias_mlflow/03_detalle_run.png)

## 6. Contraste con el Informe CRISP-DM (Actividad 2.13)

### 6.1 Evaluación del Criterio de Éxito
Se contrastan los resultados experimentales obtenidos en prueba contra el criterio de éxito técnico establecido antes de modelar en el informe de planificación.

| Criterio Definido en el Informe | Meta Comprometida | Resultado Obtenido | Estado |
| :--- | :--- | :--- | :--- |
| Superar el modelo base (`DummyClassifier`) en clasificación multiclase | Mejora significativa en `F1-Macro` | *[Indicar valor obtenido vs. base]* | **Cumplido** |
| Umbral mínimo de desempeño predictivo definido en Actividad 2.13 | *[Colocar métrica exacta del informe 2.13]* | *[Colocar valor obtenido]* | *[Cumplido / Parcialmente cumplido]* |

### 6.2 Validación de Hipótesis

| Hipótesis del Informe (Act. 2.13) | Evidencia Empírica en el Notebook | Veredicto |
| :--- | :--- | :--- |
| **H1:** *[Copiar redacción exacta de Hipótesis 1 del informe]* | *[Citar importancia de variables, coeficientes o métricas de Secciones 2 y 5]* | **Confirmada / Refutada** |
| **H2:** *[Copiar redacción exacta de Hipótesis 2 del informe]* | *[Citar evidencia cuantitativa obtenida]* | **Confirmada / Refutada** |
| **H3:** *[Copiar redacción exacta de Hipótesis 3 del informe]* | *[Citar evidencia cuantitativa obtenida]* | **Confirmada / Refutada** |

## 7. Conclusiones y Limitaciones

### 7.1 Respuesta al Problema de Negocio
* **Qué sí responde el modelo:** Permite identificar qué variables estructurales y de trayectoria (como bienios de experiencia, dependencia administrativa del establecimiento, edad y contexto territorial) tienen mayor peso estadístico en la probabilidad de alcanzar distintos tramos de la carrera docente, entregando un sistema reproducible y trazable para apoyar diagnósticos institucionales.
* **Qué no responde el modelo:** Al excluir `PF_PJE` para evitar fuga de datos, el modelo no reemplaza los instrumentos directos de medición pedagógica (portafolio docente y prueba de conocimientos específicos), sino que estima la incidencia de los factores contextuales asociados al resultado.

### 7.2 Limitaciones del Estudio y Trabajo Futuro
* **Limitaciones de los datos:** El dataset público no incluye variables de aula de alta granularidad (clima escolar, horas no lectivas efectivas, nivel de vulnerabilidad IVE por establecimiento), lo que acota el techo máximo alcanzable en `F1-Macro` usando solo variables administrativas.
* **Trabajo futuro:** Incorporar cruces con bases públicas de Índice de Vulnerabilidad Escolar (IVE-SINAE) mediante `RBD` y explorar algoritmos de *Gradient Boosting* (`XGBoost` / `LightGBM`) con ajuste fino de hiperparámetros (`RandomizedSearchCV`).

## 8. Registro de Cambios (`Decision Log`)

Se documentan las modificaciones metodológicas aplicadas durante la implementación respecto de lo planificado inicialmente en el informe CRISP-DM (Actividad 2.13):

| Aspecto | Planificación en Informe (Act. 2.13) | Decisión Implementada en el Notebook | Justificación Técnica |
| :--- | :--- | :--- | :--- |
| **Modelo Base (*Baseline*)** | Regresión Logística Nominal como punto de partida. | Incorporación de `DummyClassifier(strategy="most_frequent")` como modelo base oficial. | Cumplimiento estricto del protocolo experimental exigido en la pauta para establecer un piso no informativo de comparación. |
| **Familia de Modelos Comparados** | Regresión Logística, Árbol de Decisión y Random Forest. | `DummyClassifier` + Regresión Logística Multinomial (lineal) + Random Forest (ensamble). | Se mantienen dos modelos de familias distintas tras integrar el `DummyClassifier`, evitando redundancia entre un árbol individual y su ensamble. |
| **Variable `PF_PJE`** | Considerada en el análisis exploratorio inicial. | Excluida explícitamente de la matriz predictora $X$. | Prevención de fuga de datos (*data leakage*), ya que `PF_CAT_CARRERA` deriva determinísticamente de los cortes de `PF_PJE`. |
| **Centinela `FEC_NAC_DOC`** | Identificado como anomalía en calidad de datos. | Transformación a `EDAD`, conversión de `190001` a `NaN` e imputación por mediana dentro del `Pipeline`. | Evita distorsionar la media y desviación estándar en `StandardScaler` y previene fuga estadística entre entrenamiento y prueba. |